# مدل IFC با بردار جابجایی واقعی هر Image (به‌جای Embedding دلخواه)

## پیشینه
- **Notebook 16**: خروجی یک‌جای کل ماتریس IFC `(8,72,3,3)=15552` از یک بردار context — MAE=1.2253
- **Notebook 17**: خروجی factorized با `nn.Embedding(72, dim)` دلخواه برای هر تصویر — MAE=2.4008 (بدتر!)
- **تست‌های 18 (v1-v3)**: تأیید شد که `supercell.s2u_map` صحیح است و می‌توان بردار جابجایی
  فیزیکی واقعی هر تصویر را با گروه‌بندی اتم‌های سوپرسل بر اساس آن استخراج کرد. روی نمونه‌ی
  تستی (`Cr2AlC`)، بردار جابجایی هر تصویر بین تمام اتم‌های سلول واحد دقیقاً یکسان بود
  (انحراف استاندارد ~۱۰⁻¹⁵، یعنی صفر در حد خطای عددی)

## تغییر این نسخه
به‌جای `nn.Embedding(72, dim)` که در Notebook 17 یک نگاشت کاملاً بی‌معنی و یادگیری‌شده از
صفر بود، اینجا **بردار جابجایی واقعی fractional** هر تصویر (`Δfrac = R · L⁻¹`, که `R` بردار
جابجایی کارتزی و `L` لاتیس سوپرسل است) به‌عنوان query مدل استفاده می‌شود. این بردار:
- بین مواد مختلف قابل‌مقایسه است (چون fractional و بدون بُعد است، نه بر اساس Å خام)
- یک سیگنال فیزیکی واقعی حمل می‌کند: فاصله‌ی واقعی تصویر از سلول مرجع
- نیازی به یادگیری از صفر ندارد — فقط باید یک MLP کوچک یاد بگیرد این بردار را به IFC نگاشت کند

## استخراج بردار جابجایی (خلاصه‌ی روش تست‌شده)
1. `supercell.s2u_map`: هر اتم سوپرسل را به نماینده‌ی گروهش (در همان آرایه‌ی سوپرسل) نگاشت می‌کند
2. اتم‌ها بر اساس این مقدار در `n_atoms` گروه (هرکدام `n_images` عضو) دسته می‌شوند
3. درون هر گروه، بردار جابجایی هر عضو نسبت به نماینده‌ی گروه حساب و بر اساس norm مرتب می‌شود
   (این مرتب‌سازی، image index قابل‌مقایسه بین گروه‌های مختلف می‌سازد)
4. بردار جابجایی کارتزی به fractional (نسبت به لاتیس سوپرسل) تبدیل می‌شود

In [ ]:
!pip install -q phonopy torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [1]:
!pip install -q phonopy torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.3/781.3 kB 14.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.2/804.2 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 7.3 MB/s eta 0:00:00


In [2]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

device = torch.device('cpu')
print(f"Device: {device}")

Device: cpu


## مسیرهای داده

In [3]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = '✅' if os.path.exists(path) else '❌ یافت نشد'
    print(f"{exists}  {name}  →  {path}")

✅  FC_DIR  →  /kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C
✅  POSCAR_DIR  →  /kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar
✅  BANDS_DIR  →  /kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C
✅  ELASTIC_FILE  →  /kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json
✅  FEATURE_CSV  →  /kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv


## کشف خودکار Supercell صحیح (مستقل برای هر ماده) — بدون تغییر نسبت به Notebook 16/17

In [4]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    masses = [p['mass'] for p in data['points']]
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc

    return {
        'phonon': phonon,
        'symbols': symbols,
        'frac_coords': frac_coords,
        'lattice': lattice,
        'masses': masses,
        'supercell_dim': dim,
        'dim_match_error': err,
        'force_constants': fc,
        'all_q_points': [p['q-position'] for p in data['phonon']],
        'all_real_freqs': np.array([sorted([b['frequency'] for b in p['band']]) for p in data['phonon']]),
    }

print("✅ توابع کشف Supercell آماده شدند")

✅ توابع کشف Supercell آماده شدند


## تابع جدید: استخراج بردار جابجایی fractional واقعی هر Image

این تابع دقیقاً همان منطقی است که در تست‌های v1-v3 تأیید شد، اما برای استفاده‌ی عمومی
(هر تعداد اتم/تصویر) بسته‌بندی شده است.

In [5]:
def extract_image_displacement_vectors(phonon, n_atoms, n_images):
    """
    بردار جابجایی fractional هر تصویر سوپرسل را استخراج می‌کند.
    خروجی: آرایه‌ی (n_images, 3) -- یکسان برای تمام n_atoms اتم سلول واحد (طبق تست‌های v1-v3)
    اگر ساختار سوپرسل با فرض تست (گروه‌های مساوی + جابجایی یکسان بین گروه‌ها) مطابقت نداشت،
    None برمی‌گرداند (در آن صورت آن ماده باید رد شود).
    """
    supercell = phonon.supercell
    sc_cart = supercell.positions          # (n_supercell, 3)
    sc_lattice = supercell.cell            # (3,3) لاتیس سوپرسل واقعی
    s2u_map = supercell.s2u_map            # (n_supercell,)

    unique_reps = np.unique(s2u_map)
    if len(unique_reps) != n_atoms:
        return None

    groups = {rep: np.where(s2u_map == rep)[0] for rep in unique_reps}
    if not all(len(v) == n_images for v in groups.values()):
        return None

    sorted_displacements_per_group = []
    for rep in unique_reps:
        members = groups[rep]
        disp = sc_cart[members] - sc_cart[rep]       # (n_images, 3) کارتزی
        norms = np.linalg.norm(disp, axis=1)
        order = np.argsort(norms)
        sorted_displacements_per_group.append(disp[order])
    sorted_displacements_per_group = np.array(sorted_displacements_per_group)  # (n_atoms, n_images, 3)

    # بررسی سازگاری بین گروه‌ها (همان چک تست v3)
    std_across_groups = np.std(sorted_displacements_per_group, axis=0)  # (n_images, 3)
    if np.max(std_across_groups) > 1e-2:  # تلورانس بر اساس دقت float مشاهده‌شده در تست (~1e-14)
        return None

    mean_disp_cart = np.mean(sorted_displacements_per_group, axis=0)  # (n_images, 3)

    # تبدیل به fractional نسبت به لاتیس سوپرسل: frac = cart @ inv(lattice)
    try:
        inv_lattice = np.linalg.inv(sc_lattice)
    except np.linalg.LinAlgError:
        return None
    mean_disp_frac = mean_disp_cart @ inv_lattice  # (n_images, 3)

    return mean_disp_frac.astype(np.float32)

print("✅ تابع extract_image_displacement_vectors آماده شد")

✅ تابع extract_image_displacement_vectors آماده شد


## ساخت دیتاست کامل — بدون فیلتر اتم (شکل ثابت: 8×72×3×3) + بردار جابجایی هر ماده

In [6]:
POSCAR_DIR_PATH = Path(POSCAR_DIR)
FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}
poscar_files = {f.stem: f for f in POSCAR_DIR_PATH.glob('*.psc')}

common = sorted(set(fc_files) & set(band_yaml_files) & set(poscar_files))
print(f"تعداد مواد مشترک: {len(common)}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

تعداد مواد مشترک: 358


In [7]:
raw_samples = []
failed_phonopy = []
failed_shape = []
failed_displacement = []
failed_other = []

N_ATOMS_FIXED = 8       # طبق نتیجه‌ی بخش A، Notebook 16
N_SUPERCELL_FIXED = 72  # طبق نتیجه‌ی بخش A، Notebook 16
N_IMAGES_FIXED = N_SUPERCELL_FIXED // N_ATOMS_FIXED  # 9

for formula in tqdm(common, desc="بارگذاری مواد + استخراج بردار جابجایی"):
    try:
        with open(band_yaml_files[formula]) as f:
            quick_check = yaml.safe_load(f)
        if quick_check['natom'] != N_ATOMS_FIXED:
            failed_shape.append(formula)
            continue

        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            failed_phonopy.append(formula)
            continue

        if result['force_constants'].shape != (N_ATOMS_FIXED, N_SUPERCELL_FIXED, 3, 3):
            failed_shape.append(formula)
            continue

        disp_vectors = extract_image_displacement_vectors(
            result['phonon'], N_ATOMS_FIXED, N_IMAGES_FIXED)
        if disp_vectors is None:
            failed_displacement.append(formula)
            continue

        positions_cart = result['frac_coords'] @ result['lattice']

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           N_ATOMS_FIXED,
            'lattice':           result['lattice'].astype(np.float32),
            'positions':         positions_cart.astype(np.float32),
            'atom_elements':     result['symbols'],
            'masses':            np.array(result['masses'], dtype=np.float32),
            'force_constants':   result['force_constants'].astype(np.float32),  # (8, 72, 3, 3)
            'supercell_dim':     result['supercell_dim'],
            'image_displacements': disp_vectors,  # (9, 3) fractional -- جدید در این نسخه
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          result['all_real_freqs'].astype(np.float32),
            'phonon_obj':        result['phonon'],
        })
    except Exception as e:
        failed_other.append((formula, str(e)))

print(f"\n✅ موفق: {len(raw_samples)}")
print(f"❌ شکل نامنطبق با (8,72,3,3): {len(failed_shape)}")
print(f"❌ شکست کشف Supercell: {len(failed_phonopy)}")
print(f"❌ شکست استخراج بردار جابجایی (ناسازگاری بین گروه‌ها): {len(failed_displacement)}")
print(f"❌ خطای دیگر: {len(failed_other)}")

dims_found = Counter(s['supercell_dim'] for s in raw_samples)
print(f"\n📊 توزیع supercell_dim کشف‌شده در عمل:")
for dim, count in dims_found.most_common():
    print(f"   {dim}: {count} ماده")

بارگذاری مواد + استخراج بردار جابجایی:   0%|          | 0/358 [00:00<?, ?it/s]

/tmp/ipykernel_58/4281210925.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/usr/local/lib/python3.12/dist-packages/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()



✅ موفق: 74
❌ شکل نامنطبق با (8,72,3,3): 0
❌ شکست کشف Supercell: 0
❌ شکست استخراج بردار جابجایی (ناسازگاری بین گروه‌ها): 284
❌ خطای دیگر: 0

📊 توزیع supercell_dim کشف‌شده در عمل:
   (3, 3, 1): 74 ماده


## تقسیم Train / Val / Test (seed=42) — مشابه Notebook 16/17 برای مقایسه‌ی منصفانه

In [8]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)+-
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

Train: 59 | Val: 7 | Test: 8


## ساخت گراف (Bond + Atom) — بدون تغییر نسبت به Notebook 16/17

In [9]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"فیچرهای اتمی: {N_ATOM_FEATURES} → {feature_cols}")


def structure_to_bond_graph(positions):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)


def structure_to_atom_graph(atom_elements, positions):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

print("✅ توابع ساخت گراف آماده شدند")

فیچرهای اتمی: 7 → ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
✅ توابع ساخت گراف آماده شدند


In [10]:
bond_graphs, atom_graphs, ifc_targets, image_disp_targets = [], [], [], []

for s in tqdm(raw_samples, desc="ساخت گراف‌ها"):
    bond_graphs.append(structure_to_bond_graph(s['positions']))
    atom_graphs.append(structure_to_atom_graph(s['atom_elements'], s['positions']))
    ifc_targets.append(s['force_constants'])           # (8, 72, 3, 3)
    image_disp_targets.append(s['image_displacements']) # (9, 3) fractional -- جدید

print(f"✅ {len(bond_graphs)} نمونه گراف‌سازی شد")

ساخت گراف‌ها:   0%|          | 0/74 [00:00<?, ?it/s]

✅ 74 نمونه گراف‌سازی شد


## معماری مدل: Dual Graph GNN با Query مبتنی بر بردار جابجایی واقعی (نه Embedding دلخواه)

**تفاوت کلیدی با Notebook 17:**
- به‌جای `nn.Embedding(72, dim)` که یک نگاشت بی‌معنی از ایندکس به بردار یادگیری‌شده بود،
  اینجا بردار جابجایی fractional **واقعی** هر تصویر (`image_displacements`, شکل `(9,3)`)
  مستقیماً وارد یک MLP کوچک (`displacement_encoder`) می‌شود تا به یک query embedding تبدیل شود
- چون این بردار از فیزیک واقعی ماده می‌آید (نه از صفر یادگیری شود)، انتظار می‌رود مدل
  راحت‌تر یاد بگیرد که چگونه فاصله‌ی فیزیکی به افت‌وخیز IFC مرتبط است (مثلاً تصاویر دورتر
  باید IFC نزدیک به صفر داشته باشند — یک سیگنال که در Embedding دلخواه اصلاً وجود نداشت)

In [20]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphRealDisplacementIFCNet(nn.Module):
    """پیش‌بینی IFC با query مبتنی بر بردار جابجایی fractional واقعی هر تصویر (نه Embedding دلخواه)."""
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1,
                 n_atoms_unitcell=8, n_images=9, disp_emb_dim=64):
        super().__init__()
        self.n_atoms_unitcell = n_atoms_unitcell
        self.n_images = n_images
        hidden = 128 if n_atom_features <= 10 else 96

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        self.context_dim = 8 * hidden + hidden // 4  # بردار context سراسری ماده

# --- بخش جدید: encoder بردار جابجایی واقعی ---
        self.disp_emb_dim = disp_emb_dim
        self.displacement_encoder = nn.Sequential(
            nn.Linear(3, 32), nn.SiLU(),
            nn.Linear(32, disp_emb_dim), nn.SiLU()
        )

        per_image_in_dim = self.context_dim + disp_emb_dim
        
        # اصلاح ابعاد: 8 اتم مرجع × 8 اتم تصویر × 3 × 3 = 576
        per_image_out_dim = n_atoms_unitcell * n_atoms_unitcell * 3 * 3  

        self.ifc_head = nn.Sequential(
            nn.Linear(per_image_in_dim, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(0.15),
            nn.Linear(256, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(0.15), # کمی بزرگتر شد تا ظرفیت حفظ شود
            nn.Linear(512, per_image_out_dim)
        )
    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data, image_displacements):
        """image_displacements: (batch, n_images, 3) -- بردار جابجایی fractional واقعی هر ماده."""
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        context = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)  # (batch, context_dim)

        batch_size = context.shape[0]

        # encode بردار جابجایی واقعی هر تصویر -> (batch, n_images, disp_emb_dim)
        disp_q = self.displacement_encoder(image_displacements)  # (batch, n_images, disp_emb_dim)

        context_exp = context.unsqueeze(1).expand(-1, self.n_images, -1)  # (batch, n_images, context_dim)

# ادامه متد forward ...
        combined_per_image = torch.cat([context_exp, disp_q], dim=-1)  # (batch, n_images, context+disp_emb)
        combined_flat = combined_per_image.reshape(batch_size * self.n_images, -1)

        ifc_flat = self.ifc_head(combined_flat)  # (batch*n_images, 576)
        
        # ۱. باز کردن ابعاد به (batch, n_images, n_ref_atoms, n_img_atoms, 3, 3)
        ifc_pred = ifc_flat.view(batch_size, self.n_images, self.n_atoms_unitcell, self.n_atoms_unitcell, 3, 3)
        
        # ۲. جابجایی ابعاد برای آوردن اتم‌های مرجع به جایگاه درست: (batch, n_ref_atoms, n_images, n_img_atoms, 3, 3)
        ifc_pred = ifc_pred.permute(0, 2, 1, 3, 4, 5)
        
        # ۳. ادغام بُعد تصاویر (9) و اتم‌های داخل هر تصویر (8) برای ساختن تارگت 72
        # نتیجه نهایی: (batch, 8, 72, 3, 3)
        ifc_pred = ifc_pred.reshape(batch_size, self.n_atoms_unitcell, self.n_images * self.n_atoms_unitcell, 3, 3)
        
        return ifc_pred

print("✅ معماری DualGraphRealDisplacementIFCNet (query از بردار جابجایی واقعی) تعریف شد")

✅ معماری DualGraphRealDisplacementIFCNet (query از بردار جابجایی واقعی) تعریف شد


## Physics-Informed Loss — ASR (مشابه Notebook 16/17، برای مقایسه‌ی منصفانه)

In [21]:
def acoustic_sum_rule_loss_full(ifc_pred):
    """ASR روی ماتریس کامل (batch, 8, 72, 3, 3): مجموع روی بُعد سوپرسل (axis=2) باید صفر باشد."""
    asr = torch.sum(ifc_pred, dim=2)
    return torch.mean(asr ** 2)


LAMBDA_ASR = 0.1


def compute_batch_loss_full(model, bond_data, atom_data, ifc_target_batch, disp_batch, device):
    ifc_pred = model(bond_data, atom_data, disp_batch)  # (batch, 8, 72, 3, 3)
    ifc_true = torch.tensor(np.stack(ifc_target_batch), dtype=torch.float32, device=device)

    mse_loss = F.mse_loss(ifc_pred, ifc_true)
    asr_loss = acoustic_sum_rule_loss_full(ifc_pred)

    total_loss = mse_loss + LAMBDA_ASR * asr_loss
    return total_loss, mse_loss.item(), asr_loss.item(), ifc_pred

print(f"✅ Loss ترکیبی (بردار جابجایی واقعی): λ_asr={LAMBDA_ASR}")

✅ Loss ترکیبی (بردار جابجایی واقعی): λ_asr=0.1


## حلقه‌ی آموزش

In [22]:
def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch_full(model, indices, optimizer=None, batch_size=8, shuffle=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse, total_asr = 0., 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        ifc_list = [ifc_targets[i] for i in batch_idx]
        disp_list = [image_disp_targets[i] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)
        disp_batch = torch.tensor(np.stack(disp_list), dtype=torch.float32, device=device)  # (batch, 9, 3)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, asr_v, _ = compute_batch_loss_full(
                model, bond_batch, atom_batch, ifc_list, disp_batch, device)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        total_asr += asr_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return total_loss/n_batches, total_mse/n_batches, total_asr/n_batches

print("✅ run_epoch_full آماده شد")

✅ run_epoch_full آماده شد


In [23]:
model = DualGraphRealDisplacementIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                                          edge_dim=1, n_atoms_unitcell=8, n_images=9,
                                          disp_emb_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

EPOCHS = 500
PATIENCE = 100
BATCH_SIZE_TRAIN = 8

best_val_loss = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_mse, train_asr = run_epoch_full(
        model, train_idx, optimizer=optimizer, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loss, val_mse, val_asr = run_epoch_full(
        model, val_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, '/kaggle/working/best_real_displacement_ifc_model.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
              f"(MSE={val_mse:.4f}, ASR={val_asr:.4f}) ⭐ Best: {best_val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"⚠️ Early stop در epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\n✅ آموزش کامل شد. بهترین Val Loss: {best_val_loss:.4f}")

Epoch    0 | Train: 3.6981 | Val: 1.7249 (MSE=0.9242, ASR=8.0068) ⭐ Best: 1.7249
Epoch   10 | Train: 0.8858 | Val: 0.8277 (MSE=0.7894, ASR=0.3832) ⭐ Best: 0.8277
Epoch   20 | Train: 0.8042 | Val: 0.8028 (MSE=0.7765, ASR=0.2623) ⭐ Best: 0.8007
Epoch   30 | Train: 0.7837 | Val: 0.7952 (MSE=0.7753, ASR=0.1985) ⭐ Best: 0.7912
Epoch   40 | Train: 0.7877 | Val: 0.7873 (MSE=0.7749, ASR=0.1247) ⭐ Best: 0.7854
Epoch   50 | Train: 0.7984 | Val: 0.7840 (MSE=0.7724, ASR=0.1163) ⭐ Best: 0.7835
Epoch   60 | Train: 0.7142 | Val: 0.7228 (MSE=0.7065, ASR=0.1635) ⭐ Best: 0.7228
Epoch   70 | Train: 0.6804 | Val: 0.6736 (MSE=0.6566, ASR=0.1708) ⭐ Best: 0.6736
Epoch   80 | Train: 0.6365 | Val: 0.6089 (MSE=0.5870, ASR=0.2192) ⭐ Best: 0.6089
Epoch   90 | Train: 0.6107 | Val: 0.5790 (MSE=0.5674, ASR=0.1159) ⭐ Best: 0.5780
Epoch  100 | Train: 0.5358 | Val: 0.5245 (MSE=0.5065, ASR=0.1804) ⭐ Best: 0.5245
Epoch  110 | Train: 0.4971 | Val: 0.4499 (MSE=0.4191, ASR=0.3078) ⭐ Best: 0.4411
Epoch  120 | Train: 0.4661 |

## ارزیابی نهایی — با IFC کاملاً پیش‌بینی‌شده (بدون قرض از Ground Truth)

مقایسه‌ی مستقیم با Notebook 16 و 17 (همان داده، تقسیم‌بندی، و معیار ارزیابی).

In [25]:
model.eval()
all_freq_mae = []

with torch.no_grad():
    for i in tqdm(test_idx, desc="ارزیابی نهایی (IFC کامل، بردار جابجایی واقعی، بدون قرض)"):
        s = raw_samples[i]
        bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
        atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
        disp_batch = torch.tensor(image_disp_targets[i], dtype=torch.float32, device=device).unsqueeze(0)  # (1,9,3)

        ifc_pred_full = model(bond_batch, atom_batch, disp_batch)[0].cpu().numpy()  # (8, 72, 3, 3)

        try:
            phonon = s['phonon_obj']
            phonon.force_constants = ifc_pred_full
            phonon.run_qpoints([[0, 0, 0]])
            pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]

            true_peak = float(np.max(s['y_phonon']))
            pred_peak = float(np.max(np.abs(pred_freqs)))
            all_freq_mae.append(abs(true_peak - pred_peak))
        except Exception as e:
            continue

freq_mae = np.mean(all_freq_mae) if all_freq_mae else float('nan')
print(f"\n🔬 MAE فرکانس (IFC کامل، بردار جابجایی واقعی، بدون قرض از Ground Truth، THz): {freq_mae:.4f}")
print(f"   Notebook 11 baseline (مستقیم روی peak):              0.429")
print(f"   v1-v3 (فرمول دستی غلط):                                ~17.2")
print(f"   v4 (Phonopy واقعی، فقط بلوک مرجع، حد بالا):           0.909")
print(f"   Notebook 16 (IFC کامل، یک‌جا، خروجی 15552 بعدی):     1.2253")
print(f"   Notebook 17 (IFC کامل، factorized، embedding دلخواه): 2.4008")
print(f"   این نسخه (IFC کامل، بردار جابجایی واقعی fractional):  {freq_mae:.4f}")

ارزیابی نهایی (IFC کامل، بردار جابجایی واقعی، بدون قرض):   0%|          | 0/8 [00:00<?, ?it/s]


🔬 MAE فرکانس (IFC کامل، بردار جابجایی واقعی، بدون قرض از Ground Truth، THz): 3.1882
   Notebook 11 baseline (مستقیم روی peak):              0.429
   v1-v3 (فرمول دستی غلط):                                ~17.2
   v4 (Phonopy واقعی، فقط بلوک مرجع، حد بالا):           0.909
   Notebook 16 (IFC کامل، یک‌جا، خروجی 15552 بعدی):     1.2253
   Notebook 17 (IFC کامل، factorized، embedding دلخواه): 2.4008
   این نسخه (IFC کامل، بردار جابجایی واقعی fractional):  3.1882


/tmp/ipykernel_58/3703984900.py:17: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]


## جمع‌بندی

این نسخه فرض می‌کند که دلیل شکست Notebook 17 نبود سیگنال فیزیکی معنادار در query هر تصویر
بود (به‌جای خود ایده‌ی factorization). با جایگزینی `nn.Embedding` دلخواه با بردار جابجایی
fractional واقعی (که از خود ساختار کریستالی هر ماده می‌آید)، مدل اکنون اطلاعات صریحی درباره‌ی
فاصله‌ی فیزیکی هر تصویر از سلول مرجع دارد.

اگر MAE این نسخه به‌طور قابل‌توجهی بهتر از Notebook 17 (و حتی نزدیک یا بهتر از v4=0.909) شود،
فرضیه‌ی "کیفیت سیگنال ورودی query، نه خود factorization" تأیید می‌شود. اگر بهبودی حاصل نشد،
باید نتیجه گرفت مشکل عمیق‌تر است (مثلاً کمبود داده برای یادگیری هر نگاشتی به فضای IFC کامل)
و باید به‌سمت augmentation فیزیکی یا بازگشت به معماری v4 (با تمرکز روی بهبودهای دیگر) رفت.

### ثبت در Obsidian
نتیجه‌ی این Notebook (MAE نهایی + مقایسه با Notebook 16/17) باید در `08 - نقشه راه مقاله.md`
به‌عنوان آخرین آزمایش بهبود فاز ۳ پروژه ثبت شود.